In [1]:
import sys
sys.argv = ["moderngl_app"]  # elimină argumentele Jupyter

import numpy as np
import moderngl_window as mglw

class App(mglw.WindowConfig):
    gl_version = (3, 3)
    title = "Plasma Pulse"
    window_size = (800, 600)
    cursor = False  # Ascunde cursorul pentru un efect mai imersiv

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        # 1. SHADERELE: Aici se întâmplă magia
        self.prog = self.ctx.program(
            vertex_shader="""
                #version 330
                in vec2 in_pos;
                uniform float time;  // Primim timpul din animație
                out vec2 uv;         // Pasăm coordonatele la Fragment Shader

                void main() {
                    // Adăugăm o vibrație subtilă geometriei pe baza timpului
                    vec2 pos = in_pos;
                    pos.x += sin(time * 0.5 + in_pos.y) * 0.05;
                    pos.y += cos(time * 0.3 + in_pos.x) * 0.05;

                    gl_Position = vec4(pos, 0.0, 1.0);
                    uv = in_pos;  // Trimitem coordonatele -1..1 la FS
                }
            """,
            fragment_shader="""
                #version 330
                uniform float time;
                in vec2 uv;
                out vec4 f_color;

                void main() {
                    // Calculăm 3 unde de culoare diferite care se intersectează
                    float col_r = 0.5 + 0.5 * cos(time + uv.xyx).r;
                    float col_g = 0.5 + 0.5 * sin(time * 1.5 + uv.yxy).g;
                    float col_b = 0.5 + 0.5 * cos(time * 2.1 + uv.yyx).b;

                    // Combinăm undele pentru un efect de "plasma" pulsatorie
                    vec3 color = vec3(col_r, col_g, col_b);

                    // Adăugăm un efect de vignettare (margini mai întunecate)
                    float d = length(uv);
                    color *= 1.3 - d * 0.8;

                    f_color = vec4(color, 1.0);
                }
            """,
        )

        # 2. GEOMETRIA: Două triunghiuri care acoperă tot ecranul
        # Coordonate: [-1, -1] la [1, 1]
        verts = np.array([
            # x,   y
            -1.0, -1.0,  # Stânga-Jos
             1.0, -1.0,  # Dreapta-Jos
            -1.0,  1.0,  # Stânga-Sus
             1.0,  1.0,  # Dreapta-Sus
        ], dtype="f4")

        vbo = self.ctx.buffer(verts.tobytes())

        # Leagă bufferul și folosește TRIANGLE_STRIP pentru a desena dreptunghiul din 2 triunghiuri
        self.vao = self.ctx.simple_vertex_array(self.prog, vbo, "in_pos")

        # Obținem locația variabilei 'time' din shader
        self.time_uniform = self.prog["time"]

    def on_render(self, time: float, frame_time: float):
        self.ctx.clear(0.0, 0.0, 0.0)

        # 3. ACTUALIZAREA: Trimitem timpul curent la shader
        self.time_uniform.value = time

        # Desenăm folosind TRIANGLE_STRIP
        self.vao.render(mode=self.ctx.TRIANGLE_STRIP)

try:
    mglw.run_window_config(App, args=[])
except KeyboardInterrupt:
    print("Program oprit manual.")

2026-03-03 14:30:14,416 - moderngl_window - INFO - Attempting to load window class: moderngl_window.context.pyglet.Window
2026-03-03 14:30:14,737 - moderngl_window.context.base.window - INFO - Context Version:
2026-03-03 14:30:14,739 - moderngl_window.context.base.window - INFO - ModernGL: 5.12.0
2026-03-03 14:30:14,741 - moderngl_window.context.base.window - INFO - vendor: Intel
2026-03-03 14:30:14,742 - moderngl_window.context.base.window - INFO - renderer: Mesa Intel(R) Iris(R) Xe Graphics (ADL GT2)
2026-03-03 14:30:14,743 - moderngl_window.context.base.window - INFO - version: 4.6 (Core Profile) Mesa 25.0.7-0ubuntu0.24.04.2
2026-03-03 14:30:14,744 - moderngl_window.context.base.window - INFO - python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
2026-03-03 14:30:14,745 - moderngl_window.context.base.window - INFO - platform: linux
2026-03-03 14:30:14,746 - moderngl_window.context.base.window - INFO - code: 460
2026-03-03 14:30:23,783 - moderngl_window - INFO - Duration: 9.02s 